# D5 Climate Risk Workflow — GeoPrompt

Climate zone risk scoring, raster algebra, and adaptation scenario analysis.


In [4]:
from __future__ import annotations

import json, os
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import geoprompt as gp

_ROOT = Path.cwd()
if (_ROOT / "examples" / "notebooks").exists():
    OUTPUT_DIR = _ROOT / "examples" / "notebooks" / "geoprompt" / "outputs"
else:
    OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALLOW_LIVE_API = os.getenv("GEOPROMPT_ALLOW_LIVE_API", "0") == "1"


def fetch_json(url, fallback):
    if not ALLOW_LIVE_API:
        return fallback
    try:
        req = Request(url, headers={"User-Agent": "geoprompt-notebook/2.0"})
        with urlopen(req, timeout=6) as r:
            return json.loads(r.read().decode("utf-8"))
    except (URLError, TimeoutError, ValueError):
        return fallback


def fetch_first_json(urls, validator, fallback):
    for url in urls:
        payload = fetch_json(url, None)
        if payload is not None and validator(payload):
            return payload, url, True
    return fallback, "fallback", False


def make_point(x: float, y: float):
    return {"type": "Point", "coordinates": [float(x), float(y)]}


def frame_from_xy(data: dict[str, list], xs: list[float], ys: list[float], crs: str = "EPSG:4326"):
    columns = list(data.keys())
    rows = []
    for i, (x, y) in enumerate(zip(xs, ys)):
        row = {col: data[col][i] for col in columns}
        row["geometry"] = make_point(x, y)
        rows.append(row)
    return gp.GeoPromptFrame.from_records(rows, crs=crs)


def frame_preview(frame: gp.GeoPromptFrame, columns: list[str], limit: int | None = None):
    rows = frame.to_records()
    if limit is not None:
        rows = rows[:limit]
    return pd.DataFrame([{col: row.get(col) for col in columns} for row in rows], columns=columns)


print(f"geoprompt {gp.__version__} ready")

geoprompt 0.2.0 ready


## Section A: Pull Data Sources


In [2]:
climate = {"features": [{"id": "fallback-climate"}]}
weather = {"properties": {"forecast": "fallback"}}
forecast = {"hourly": {"temperature_2m": [0.0]}}

climate, cl_src, cl_live = fetch_first_json(
    [
        "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson",
        "https://api.github.com/repos/ecmwf/cdsapi",
    ],
    lambda d: isinstance(d, dict) and bool(d.get("features") or d.get("id")),
    climate,
)
weather, wx_src, wx_live = fetch_first_json(
    [
        "https://api.weather.gov/points/39.00,-98.00",
        "https://api.weather.gov/points/38.90,-77.03",
    ],
    lambda d: isinstance(d, dict) and bool(d.get("properties", {}).get("forecast")),
    weather,
)
forecast, fc_src, fc_live = fetch_first_json(
    [
        "https://api.open-meteo.com/v1/forecast?latitude=39.00&longitude=-98.00&hourly=temperature_2m&forecast_days=1",
        "https://api.open-meteo.com/v1/forecast?latitude=38.90&longitude=-77.03&hourly=temperature_2m&forecast_days=1",
    ],
    lambda d: isinstance(d, dict) and len(d.get("hourly", {}).get("temperature_2m", [])) > 0,
    forecast,
)

climate_count = len(climate.get("features", [])) if isinstance(climate, dict) else 0
if climate_count == 0 and isinstance(climate, dict) and climate.get("id"):
    climate_count = 1
print(f"Climate records: {climate_count} | live={cl_live} | source={cl_src}")
print(f"NOAA forecast exists: {bool(weather.get('properties', {}).get('forecast'))} | live={wx_live} | source={wx_src}")
print(f"Open-Meteo hourly points: {len(forecast.get('hourly', {}).get('temperature_2m', []))} | live={fc_live} | source={fc_src}")


Climate records: 1 | live=False | source=fallback
NOAA forecast exists: True | live=False | source=fallback
Open-Meteo hourly points: 1 | live=False | source=fallback


## Section B: Spatial Analysis


In [5]:
zones_data = {
    "zone_id":     ["Z1",  "Z2",  "Z3",  "Z4",  "Z5"],
    "heat_index":  [0.78,  0.62,  0.85,  0.71,  0.90],
    "drought_idx": [0.65,  0.72,  0.55,  0.80,  0.45],
    "sea_level":   [0.30,  0.15,  0.80,  0.20,  0.60],
    "pop_density": [1200,  850,   3400,  550,   2100],
}

lons = [-98.10, -97.80, -97.50, -98.40, -97.65]
lats = [ 39.20,  38.90,  39.40,  38.70,  39.60]

gdf = frame_from_xy(zones_data, lons, lats, crs="EPSG:4326")
gdf["composite_risk"] = [
    round(heat * 0.35 + drought * 0.35 + sea * 0.30, 4)
    for heat, drought, sea in zip(gdf["heat_index"], gdf["drought_idx"], gdf["sea_level"])
]
gdf["risk_tier"] = ["HIGH" if r > 0.65 else ("MED" if r > 0.45 else "LOW") for r in gdf["composite_risk"]]

print("Climate zone risk scores:")
print(frame_preview(gdf, ["zone_id", "composite_risk", "risk_tier"]).to_string(index=False))


# Adaptation sites
adapt_gdf = frame_from_xy(
    {"site_id": ["AS1", "AS2"], "capacity_mw": [120, 90]},
    [-97.90, -98.20],
    [39.10, 39.30],
    crs="EPSG:4326",
)


# 1. Buffer: climate impact zones (50km)
gdf_proj = gdf.to_crs("EPSG:3857")
buf_zones = gdf_proj.buffer(50000).to_crs("EPSG:4326")
print(f"Climate impact buffers (50km): {len(buf_zones)}")


# 2. Nearest join: assign zones to adaptation sites (km)
zones_m = gdf.to_crs("EPSG:3857")
sites_m = adapt_gdf.to_crs("EPSG:3857")
nearest_m = zones_m.nearest_join(sites_m, how="left", rsuffix="site")
nearest = nearest_m.to_crs("EPSG:4326")
nearest["dist_km"] = [None if d is None else d / 1000.0 for d in nearest_m["distance_site"]]

print("\nNearest adaptation site per zone (km):")
print(frame_preview(nearest, ["zone_id", "site_id", "dist_km"]).to_string(index=False))


# 3. Spatial join: zones within adaptation site coverage
adapt_buffers = adapt_gdf.select_columns(["site_id"]).to_crs("EPSG:3857").buffer(100000).to_crs("EPSG:4326")
covered = gdf.spatial_join(adapt_buffers, how="inner", predicate="within", rsuffix="adapt")
print(f"\nZones within 100km of adaptation sites: {list(covered['zone_id'])}")


# 4. Dissolve by risk tier
dissolved = (
    pd.DataFrame(gdf.to_records())
    .groupby("risk_tier", as_index=True)
    .agg({"composite_risk": "mean", "pop_density": "sum"})
)
print("\nDissolved by risk tier:")
print(dissolved[["composite_risk", "pop_density"]].to_string())


# 5. High-risk filter
high_risk = gdf.where([value > 0.65 for value in gdf["composite_risk"]])
print(f"\nHigh-risk zones: {list(high_risk['zone_id'])}")


# 6. Bounding box: northern band
north = gdf.cx[-98.50:-97.40, 39.30:39.70]
print(f"Zones in northern band: {list(north['zone_id'])}")


# 7. Overlay: zone buffer intersections with risk polygon
risk_poly = gp.GeoPromptFrame.from_records(
    [{"id": 1, "geometry": {"type": "Polygon", "coordinates": [[(-98.20, 39.00), (-97.40, 39.00), (-97.40, 39.70), (-98.20, 39.70), (-98.20, 39.00)]]}}],
    crs="EPSG:4326",
)
buf_gdf = gdf.select_columns(["zone_id"]).to_crs("EPSG:3857").buffer(30000).to_crs("EPSG:4326")
clipped = buf_gdf.clip(risk_poly)
print(f"Buffer areas intersecting risk polygon: {len(clipped)}")


gdf.to_file(str(OUTPUT_DIR / "d5-gp-zones.geojson"), driver="GeoJSON")
print("\nWrote d5-gp-zones.geojson")


# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

zone_rows = gdf.to_records()
zone_coords = [row["geometry"]["coordinates"] for row in zone_rows]
zone_x = [coord[0] for coord in zone_coords]
zone_y = [coord[1] for coord in zone_coords]
scatter = axes[0].scatter(zone_x, zone_y, c=gdf["composite_risk"], cmap="YlOrRd", s=160)
fig.colorbar(scatter, ax=axes[0], label="Composite Risk")
for row in zone_rows:
    x, y = row["geometry"]["coordinates"]
    axes[0].annotate(
        f"{row['zone_id']}\n({row['composite_risk']:.2f})",
        (x, y),
        textcoords="offset points",
        xytext=(4, 4),
        fontsize=9,
    )

adapt_rows = adapt_gdf.to_records()
adapt_coords = [row["geometry"]["coordinates"] for row in adapt_rows]
axes[0].scatter(
    [coord[0] for coord in adapt_coords],
    [coord[1] for coord in adapt_coords],
    color="blue",
    s=220,
    marker="*",
    label="Adaptation Site",
)
axes[0].set_title("Climate Risk Zones (stars=adaptation sites)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

gdf_s = gdf.sort_values("composite_risk", descending=True)
bar_colors = ["#c0392b" if t == "HIGH" else ("#e67e22" if t == "MED" else "#27ae60") for t in gdf_s["risk_tier"]]
axes[1].barh(gdf_s["zone_id"], gdf_s["composite_risk"], color=bar_colors)
axes[1].set_xlabel("Composite Climate Risk")
axes[1].set_title("Zone Risk Ranking")
axes[1].grid(True, axis="x", alpha=0.3)

plt.suptitle("D5 Climate: GeoPrompt Spatial Analysis", fontweight="bold")
plt.tight_layout()
plt.show()

# Basemap snapshot (real tiled basemap, saved as HTML)
try:
    import folium

    label_candidates = ["node", "stand_id", "asset_id", "zone_id"]
    label_col = next((c for c in label_candidates if c in gdf.columns), gdf.columns[0])
    centroids = gdf.geom.centroid()
    center_lat = float(np.mean([pt[1] for pt in centroids]))
    center_lon = float(np.mean([pt[0] for pt in centroids]))
    fmap = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles="CartoDB positron")

    for row in gdf.to_records():
        lon, lat = row["geometry"]["coordinates"]
        folium.CircleMarker(
            location=[float(lat), float(lon)],
            radius=6,
            color="#1d4ed8",
            fill=True,
            fill_opacity=0.85,
            popup=f"{label_col}: {row[label_col]}",
        ).add_to(fmap)

    optional_marker_specs = [
        ("demand_gdf", "demand_id", "blue", "bolt", "fa"),
        ("stations_gdf", "station_id", "green", "tree", "fa"),
        ("incidents_gdf", "inc_id", "red", "warning-sign", "glyphicon"),
        ("adapt_gdf", "site_id", "purple", "leaf", "fa"),
    ]

    for frame_name, id_key, color, icon, prefix in optional_marker_specs:
        frame = globals().get(frame_name)
        if frame is None:
            continue
        for row in frame.to_records():
            lon, lat = row["geometry"]["coordinates"]
            folium.Marker(
                location=[float(lat), float(lon)],
                icon=folium.Icon(color=color, icon=icon, prefix=prefix),
                popup=f"{id_key}: {row.get(id_key, 'n/a')}"
            ).add_to(fmap)

    map_path = OUTPUT_DIR / "d5-gp-basemap.html"
    fmap.save(str(map_path))
    print(f"Wrote basemap snapshot: {map_path.name}")
    from IPython.display import display
    display(fmap)
except Exception as exc:
    print(f"Basemap snapshot skipped: {exc}")

Climate zone risk scores:
zone_id  composite_risk risk_tier
     Z1          0.5905       MED
     Z2          0.5140       MED
     Z3          0.7300      HIGH
     Z4          0.5885       MED
     Z5          0.6525      HIGH
Climate impact buffers (50km): 5

Nearest adaptation site per zone (km):
zone_id site_id   dist_km
     Z1     AS2 18.181394
     Z2     AS1 30.735100
     Z3     AS1 61.988051
     Z4     AS1 79.822832
     Z5     AS2 74.960385

Zones within 100km of adaptation sites: ['Z1', 'Z1', 'Z2', 'Z2', 'Z3', 'Z3', 'Z4', 'Z4', 'Z5', 'Z5']

Dissolved by risk tier:
           composite_risk  pop_density
risk_tier                             
HIGH             0.691250         5500
MED              0.564333         2600

High-risk zones: ['Z3', 'Z5']
Zones in northern band: ['Z3', 'Z5']
Buffer areas intersecting risk polygon: 4

Wrote d5-gp-zones.geojson
Wrote basemap snapshot: d5-gp-basemap.html


C:\Users\Matt\AppData\Local\Temp\ipykernel_16496\1765827794.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section C: Scenario Comparison


In [4]:
baseline   = {"annual_loss_musd": 58.0, "resilience_index": 0.44, "service_reliability": 0.82}

adaptation = {"annual_loss_musd": 32.0, "resilience_index": 0.68, "service_reliability": 0.91}



metrics = list(baseline.keys())

delta = [round((adaptation[m] - baseline[m]) / abs(baseline[m]) * 100, 1) for m in metrics]



fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x = range(len(metrics))

width = 0.38

axes[0].bar([i-width/2 for i in x], [baseline[m] for m in metrics], width=width, label="Baseline", color="#94a3b8")

axes[0].bar([i+width/2 for i in x], [adaptation[m] for m in metrics], width=width, label="Adaptation", color="#2563eb")

axes[0].set_xticks(list(x)); axes[0].set_xticklabels(metrics, rotation=15)

axes[0].set_title("Baseline vs Adaptation"); axes[0].legend(); axes[0].grid(True, axis="y", alpha=0.3)



bar_colors = ["#27ae60" if d > 0 else "#c0392b" for d in delta]

axes[1].barh(metrics, delta, color=bar_colors)

axes[1].axvline(0, color="#555", linewidth=1)

axes[1].set_xlabel("% Change vs Baseline"); axes[1].set_title("Improvement (positive=better)")

axes[1].grid(True, axis="x", alpha=0.3)

plt.suptitle("D5 Climate (GeoPrompt): Adaptation Scenario Analysis", fontweight="bold")

plt.tight_layout(); plt.show()



(OUTPUT_DIR / "d5-gp-complex.json").write_text(

    json.dumps({"baseline": baseline, "adaptation": adaptation}, indent=2, default=str), encoding="utf-8"

)

print("Wrote d5-gp-complex.json")

print("Delta per metric:", dict(zip(metrics, delta)))


Wrote d5-gp-complex.json
Delta per metric: {'annual_loss_musd': -44.8, 'resilience_index': 54.5, 'service_reliability': 11.0}


C:\Users\Matt\AppData\Local\Temp\ipykernel_22324\683266249.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
